# CuttleFish Notebook Workflow

This notebook is the primary interactive entrypoint. Run cells one by one and tune figure settings directly in each plotting cell.


In [ ]:
# %% Cell 1: Imports and paths
import math
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from CuttleFishModel.core import PARAMS, simulate
from CuttleFishModel.metrics import (
    nearest_neighbor_distances,
    pair_correlation_like,
    pigment_points,
    summarize_nnd,
    timeline_to_dataframe,
)
from CuttleFishModel.controls import (
    MODE_COLORS,
    MODE_LABELS,
    run_ablation,
    run_random_development_matched,
    run_random_mask,
    scan_parameter_landscape,
    summarize_result,
)
from CuttleFishModel.render import (
    apply_plot_style,
    draw_color_composition,
    draw_cv_bar,
    draw_frame,
    draw_fraction_stack,
    draw_nnd_distribution,
    draw_pair_correlation,
    draw_parameter_heatmap,
    draw_timeline_slices,
)

apply_plot_style()

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
REPORT_DIR = RESULTS_DIR
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, REPORT_DIR


In [ ]:
# %% Cell 2: Parameters
params = PARAMS.copy()

# Editable key parameters
params["yellow_duration"] = 28
params["red_duration"] = 3
params["absolute_min_distance"] = 2.6
params["target_gap_to_black"] = 5.0
params["growth_motion_strength"] = 1.0
params["base_birth_rate"] = 0.82
params["n_steps"] = 170
params["seed"] = 7

params


In [ ]:
# %% Cell 3: Run self-organized model
timeline, final_step = simulate(params)
final_frame = timeline[final_step]

print(f"final_step = {final_step}")
print(f"pigment_count = {final_frame.pigment_count}")
print(f"yellow / red / black = {final_frame.yellow_count} / {final_frame.red_count} / {final_frame.black_count}")
print(f"CV_NND = {final_frame.nnd_cv:.6f}")

final_frame


In [ ]:
# %% Cell 4: Preview final frame
figsize = (6.4, 6.4)
dpi = 220
preview_title = (
    f"step {final_step} | N={final_frame.pigment_count} | "
    f"Y={final_frame.yellow_count} R={final_frame.red_count} B={final_frame.black_count}"
)
preview_path = REPORT_DIR / "final_frame_preview.png"

fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
draw_frame(ax, final_frame, params, title=preview_title)
fig.savefig(preview_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(preview_path)


In [ ]:
# %% Optional: Save GIF if needed
frame_stride = params["frame_stride"]
gif_duration_ms = params["gif_duration_ms"]
gif_path = RESULTS_DIR / "01_intercalated_development.gif"

frame_steps = list(range(0, final_step + 1, frame_stride))
if frame_steps[-1] != final_step:
    frame_steps.append(final_step)

with TemporaryDirectory() as tmp_dir_name:
    tmp_dir = Path(tmp_dir_name)
    frame_paths = []
    for step in frame_steps:
        frame = timeline[step]
        fig, ax = plt.subplots(figsize=(6.2, 6.2), constrained_layout=True)
        draw_frame(
            ax,
            frame,
            params,
            title=(
                f"step {frame.step} | skin={frame.skin_area} | pigments={frame.pigment_count} | "
                f"Y={frame.yellow_count} R={frame.red_count} B={frame.black_count}"
            ),
        )
        frame_path = tmp_dir / f"frame_{step:04d}.png"
        fig.savefig(frame_path, dpi=180)
        plt.close(fig)
        frame_paths.append(frame_path)

    images = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as img:
            images.append(img.convert("P", palette=Image.ADAPTIVE))
    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=gif_duration_ms,
        loop=0,
        disposal=2,
    )

print(gif_path)


In [ ]:
# %% Cell 5: Run matched controls
self_summary = summarize_result({
    "mode": "self",
    "seed": params["seed"],
    "final_frame": final_frame,
    "birth_rate_scale": 1.0,
    "switches": {
        "use_repulsion": True,
        "use_gap_birth": True,
        "use_growth_displacement": True,
    },
})

random_dev_result = run_random_development_matched(params, target_N=final_frame.pigment_count, seed=params["seed"])
random_mask_result = run_random_mask(final_frame, target_N=final_frame.pigment_count, seed=params["seed"] + 400)

random_dev_summary = summarize_result(random_dev_result)
random_mask_summary = summarize_result(random_mask_result)
summary_df = pd.DataFrame([self_summary, random_dev_summary, random_mask_summary])

print(summary_df[["model", "N", "CV_NND", "yellow_count", "red_count", "black_count"]])
print(f"random-development birth_rate_scale = {random_dev_result['birth_rate_scale']:.6f}")

summary_df


In [ ]:
# %% Cell 6: Fig1 development slices
fig1_path = REPORT_DIR / "Fig1_development_slices_self_vs_random.png"
figsize = (19.0, 7.2)
dpi = 220

shared_steps = [0, 40, 80, 120]
shared_steps = [step for step in shared_steps if step <= final_step]
while len(shared_steps) < 4:
    shared_steps.append(min(final_step, shared_steps[-1] + 1 if shared_steps else 0))
self_steps = shared_steps[:4] + [final_step]
random_steps = shared_steps[:4] + [random_dev_result["final_step"]]

fig, axes = plt.subplots(2, 5, figsize=figsize, constrained_layout=True)
draw_timeline_slices(axes[0], timeline, self_steps, params, row_label="Self-organized")
draw_timeline_slices(
    axes[1],
    random_dev_result["timeline"],
    random_steps,
    params,
    row_label="Random-development matched",
)
fig.suptitle("Development Slices: Self-Organized vs Random-Development Matched", y=1.02, fontsize=17, fontweight="bold")
fig.savefig(fig1_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig1_path)


In [ ]:
# %% Cell 7: Fig2 NND distribution and CV
fig2_path = REPORT_DIR / "Fig2_nnd_distribution_and_cv.png"
summary_csv_path = REPORT_DIR / "random_vs_self_summary.csv"
figsize = (12.5, 5.2)
dpi = 220

summary_df.to_csv(summary_csv_path, index=False)

nnd_map = {
    "self": nearest_neighbor_distances(pigment_points(final_frame)),
    "random_development_matched": nearest_neighbor_distances(pigment_points(random_dev_result["final_frame"])),
    "random_mask": nearest_neighbor_distances(pigment_points(random_mask_result["final_frame"])),
}
labels = {
    "self": MODE_LABELS["self"],
    "random_development_matched": MODE_LABELS["random_development_matched"],
    "random_mask": MODE_LABELS["random_mask"],
}
colors = {
    "self": MODE_COLORS["self"],
    "random_development_matched": MODE_COLORS["random_development_matched"],
    "random_mask": MODE_COLORS["random_mask"],
}
bins = np.linspace(0.0, 10.0, 28)
bar_order = ["self", "random_development_matched", "random_mask"]
bar_labels = ["self", "random-dev", "random-mask"]
bar_colors = [colors[key] for key in bar_order]

fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
draw_nnd_distribution(axes[0], nnd_map, labels, colors, bins=bins, linewidth=2.2, title="NND Distribution", legend_loc="upper left")
draw_cv_bar(
    axes[1],
    summary_df,
    bar_order,
    bar_labels,
    bar_colors,
    value_col="CV_NND",
    ylabel="CV of nearest-neighbour distance",
    title="CV_NND Comparison",
    value_label_fmt="{:.3f}",
)
fig.savefig(fig2_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig2_path)
print(summary_csv_path)


In [ ]:
# %% Cell 8: Fig3 color composition over time
fig3_path = REPORT_DIR / "Fig3_color_composition_over_time.png"
metrics_csv_path = REPORT_DIR / "metrics_over_time.csv"
figsize = (13.0, 5.2)
dpi = 220

metrics_df = timeline_to_dataframe(timeline)
metrics_df.to_csv(metrics_csv_path, index=False)

line_colors = {"yellow": "#ffcc33", "red": "#c94b23", "black": "#171411"}
line_widths = {"yellow": 2.5, "red": 2.5, "black": 2.8}

fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
draw_color_composition(axes[0], metrics_df, fraction=False, colors=line_colors, linewidths=line_widths, title="Colour Composition Over Time")
draw_color_composition(axes[1], metrics_df, fraction=True, colors=line_colors, linewidths=line_widths, title="Colour Fractions Over Time")
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.03))
fig.savefig(fig3_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig3_path)
print(metrics_csv_path)


In [ ]:
# %% Cell 9: Fig4 parameter heatmap
fig4_path = REPORT_DIR / "Fig4_parameter_phase_heatmap.png"
parameter_scan_csv_path = REPORT_DIR / "parameter_scan.csv"
figsize = (13.4, 5.6)
dpi = 220

absolute_min_values = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
target_gap_values = [3.0, 4.0, 5.0, 6.0, 7.0]
n_replicates = 3
replicate_seeds = [params["seed"] + 10 * i for i in range(n_replicates)]

scan_df = scan_parameter_landscape(
    params,
    absolute_min_distance_values=absolute_min_values,
    target_gap_values=target_gap_values,
    seeds=replicate_seeds,
)
scan_df.to_csv(parameter_scan_csv_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)
image1 = draw_parameter_heatmap(axes[0], scan_df, value_col="final_cv_nnd", cmap="magma_r", title="Mean Final CV_NND")
cbar1 = fig.colorbar(image1, ax=axes[0], shrink=0.92)
cbar1.set_label("CV_NND")
image2 = draw_parameter_heatmap(axes[1], scan_df, value_col="order_score", cmap="viridis", title="Mean Order Score")
cbar2 = fig.colorbar(image2, ax=axes[1], shrink=0.92)
cbar2.set_label("Order score")
fig.suptitle("Local-rule Parameter Landscape of Spacing Order", y=1.02, fontsize=17, fontweight="bold")
fig.savefig(fig4_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig4_path)
print(parameter_scan_csv_path)


In [ ]:
# %% Cell 10: Fig5 pair correlation
fig5_path = REPORT_DIR / "Fig5_pair_correlation.png"
figsize = (8.0, 5.2)
dpi = 220
r_max = 18.0
bins = np.linspace(0.0, r_max, 31)

curve_map = {
    "self": pair_correlation_like(pigment_points(final_frame), bins=bins),
    "random_mask": pair_correlation_like(pigment_points(random_mask_result["final_frame"]), bins=bins),
}
labels = {
    "self": MODE_LABELS["self"],
    "random_mask": MODE_LABELS["random_mask"],
}
colors = {
    "self": MODE_COLORS["self"],
    "random_mask": MODE_COLORS["random_mask"],
}

fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
draw_pair_correlation(ax, curve_map, labels, colors, linewidth=2.4, title="Pair-Correlation-Like Curve", shade_until=3.0)
fig.savefig(fig5_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig5_path)


In [ ]:
# %% Cell 11: Ablation runs
ablation_csv_path = REPORT_DIR / "ablation_summary.csv"

full_self_result = {
    "mode": "self",
    "seed": params["seed"],
    "timeline": timeline,
    "final_step": final_step,
    "final_frame": final_frame,
    "birth_rate_scale": 1.0,
    "switches": {
        "use_repulsion": True,
        "use_gap_birth": True,
        "use_growth_displacement": True,
    },
    "params": params,
}
no_repulsion_result = run_ablation(params, mode="no_repulsion", target_N=final_frame.pigment_count, seed=params["seed"])
no_gap_birth_result = run_ablation(params, mode="no_gap_birth", target_N=final_frame.pigment_count, seed=params["seed"])
no_growth_displacement_result = run_ablation(params, mode="no_growth_displacement", target_N=final_frame.pigment_count, seed=params["seed"])

ablation_results = [
    full_self_result,
    no_repulsion_result,
    no_gap_birth_result,
    no_growth_displacement_result,
    random_dev_result,
    random_mask_result,
]

ablation_summary_df = pd.DataFrame([summarize_result(result) for result in ablation_results])
ablation_summary_df.to_csv(ablation_csv_path, index=False)

print(ablation_summary_df[["model", "N", "CV_NND", "yellow_count", "red_count", "black_count"]])
print(ablation_csv_path)

ablation_summary_df


In [ ]:
# %% Cell 12: Fig6 ablation summary
fig6_path = REPORT_DIR / "Fig6_ablation_summary.png"
figsize = (16.0, 5.6)
dpi = 220

order = [
    "self",
    "no_repulsion",
    "no_gap_birth",
    "no_growth_displacement",
    "random_development_matched",
    "random_mask",
]
labels = ["full", "no-rep", "no-gap", "no-move", "rand-dev", "rand-mask"]
colors = [MODE_COLORS[key] for key in order]

fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)
draw_cv_bar(axes[0], ablation_summary_df, order, labels, colors, value_col="CV_NND", ylabel="CV_NND", title="CV_NND", value_label_fmt="{:.3f}")
draw_cv_bar(axes[1], ablation_summary_df, order, labels, colors, value_col="N", ylabel="Pigment count", title="Pigment count", value_label_fmt="{:.0f}")
axes[0].tick_params(axis="x", rotation=25)
axes[1].tick_params(axis="x", rotation=25)
draw_fraction_stack(axes[2], ablation_summary_df, order, labels)
fig.suptitle("Ablation Summary", y=1.02, fontsize=17, fontweight="bold")
fig.savefig(fig6_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig6_path)


In [ ]:
# %% Cell 13: Fig7 ablation final patterns
fig7_path = REPORT_DIR / "Fig7_ablation_final_patterns.png"
figsize = (12.0, 8.0)
dpi = 220

pattern_results = [
    full_self_result,
    no_repulsion_result,
    no_gap_birth_result,
    no_growth_displacement_result,
    random_dev_result,
    random_mask_result,
]

fig, axes = plt.subplots(2, 3, figsize=figsize, constrained_layout=True)
for ax, result in zip(axes.flat, pattern_results):
    frame = result["final_frame"]
    draw_frame(
        ax,
        frame,
        params,
        title=f"{MODE_LABELS[result['mode']]}\nN={frame.pigment_count} | CV={frame.nnd_cv:.3f}",
    )
fig.suptitle("Final Patterns Across Mechanism Ablations", y=1.02, fontsize=17, fontweight="bold")
fig.savefig(fig7_path, dpi=dpi, bbox_inches="tight")
plt.show()

print(fig7_path)


In [ ]:
# %% Cell 14: Export markdown summary
summary_md_path = REPORT_DIR / "model_math_summary.md"

self_row = summary_df.set_index("model").loc["self"]
rand_row = summary_df.set_index("model").loc["random_development_matched"]
mask_row = summary_df.set_index("model").loc["random_mask"]
no_rep_row = ablation_summary_df.set_index("model").loc["no_repulsion"]
no_gap_row = ablation_summary_df.set_index("model").loc["no_gap_birth"]
no_move_row = ablation_summary_df.set_index("model").loc["no_growth_displacement"]

summary_text = f"""# 模型数学摘要

## 1. 状态变量

二维格点元胞自动机状态记为 $S_{ij}(t) \in \{0,1,2,3,4\}$：

- 0：空白区域
- 1：普通皮肤细胞
- 2：黄色新生色素细胞
- 3：红色过渡态色素细胞
- 4：黑色成熟色素细胞

## 2. 皮肤生长与出生逻辑

皮肤从中心小区域向外扩张，出生概率可概括为：

$$
P_{birth} = \beta \cdot I(d_{all}) \cdot G(d_{black}) \cdot B(d_{boundary})
$$

其中：

- $I(d_{all})$：短程抑制与最小间距
- $G(d_{black})$：对成熟黑色阵列空隙的偏好
- $B(d_{boundary})$：边界惩罚

## 3. 颜色成熟

- yellow_duration = {params['yellow_duration']}
- red_duration = {params['red_duration']}

## 4. 匹配随机对照

- self: N = {int(self_row['N'])}, CV_NND = {self_row['CV_NND']:.4f}
- random-development matched: N = {int(rand_row['N'])}, CV_NND = {rand_row['CV_NND']:.4f}
- random-mask matched: N = {int(mask_row['N'])}, CV_NND = {mask_row['CV_NND']:.4f}

## 5. Ablation

- no_repulsion: N = {int(no_rep_row['N'])}, CV_NND = {no_rep_row['CV_NND']:.4f}
- no_gap_birth: N = {int(no_gap_row['N'])}, CV_NND = {no_gap_row['CV_NND']:.4f}
- no_growth_displacement: N = {int(no_move_row['N'])}, CV_NND = {no_move_row['CV_NND']:.4f}

## 6. 图解释

- Fig1：self 与 matched random-development 的发育切片比较
- Fig2：NND 分布与 CV_NND 对比
- Fig3：yellow / red / black 随时间变化
- Fig4：局部规则参数景观
- Fig5：短距离 pair-density dip
- Fig6/Fig7：ablation 对空间秩序的影响
"""

summary_md_path.write_text(summary_text, encoding="utf-8")
print(summary_md_path)